# 91 · Appendix: can the SBI model identification be trusted?

**What this shows.** The checks behind notebook 11's calls, as thesis tables and figures:

1. the trained networks against the current code and settings (a network trained on a different simulator or
   statistic vector would condition on the wrong thing);
2. calibration of each network (simulation-based calibration, Talts et al. 2018) and the coverage of its posterior
   intervals, at an animal's full data (5 sessions) and at the CV halves the model comparison conditions on (2 and 3);
3. parameter recovery from the same draws: error and bias relative to the prior width, R², posterior contraction and
   z-scores (Schad, Betancourt & Vasishth 2021);
4. model recovery: 50 BE + 50 SC synthetic animals, each copying a real animal's session count and lengths and passing
   the real session rule (accuracy ≥ 0.70 in every session), run with exactly the real run's procedure (grid search at
   16 seeds, SBI at 64 repeats, the same folds and consensus);
5. the real animals: where their posteriors put σ_percep relative to the grid search's range, and the calls of the
   October run next to the current one.

**What produced it.** One validation run, `model_identification/synthetic_uniform_k50/<run_id>` (commands in
`docs/runs.md`, "SBI validation"): `sc-make-synthetic-cohort … --shape-from all-mice/<id> --accuracy-rule`, the grid
search and SBI conditioning on that cohort, `sc-consensus`, `sc-sbi-sbc`, `sc-sbi-check` (current and October networks),
then `sc-validation-tables`, which writes every table read here into `<run>/tables/`. Fetch with
`sc-fetch results model_identification/synthetic_uniform_k50`. This notebook reads tables only; it is not run by CI.

**How to read the calls.** Each voter's p-value comes from repeats that vary simulation noise and posterior samples on a
fixed data split, so it says the call is stable under that noise, not that the data favour it. The synthetic confusion
matrices are therefore the operating characteristic to quote next to every real call: how often this exact procedure
calls the right model, the wrong one, or none, for animals like ours.

In [ ]:
from nb_setup import *
from sound_categorisation.plotting.cv import plot_confusion
from sound_categorisation.plotting.sbi_validation import (plot_contraction, plot_coverage, plot_recovery_scatter,
                                                          plot_sbc_ecdf_diff)
banner()
run = open_run('model_identification', VALIDATION_COHORT)
T = load_validation_tables(run)
print('tables:', ', '.join(sorted(T)) or 'none -- run sc-validation-tables on this run')


def pick(df, **kw):
    for k, v in kw.items():
        df = df[df[k] == v]
    return df

## 1 · The networks against the current code

`sc-sbi-check` compares each saved network's configuration, parameter order and trained prior with the current settings,
and the median of every feature of 5,000 fresh simulations with the median stored at training (|z| > 4 flags drift in the
model cores, the stimulus sampler or the statistics). `ok` False marks a difference. The October networks are expected
to differ in `N`, `design_features` and `n_simulations` (they were trained on 4–6 sessions, without the data-amount
inputs, on 50k / 100k simulations); anything else differing, a fingerprint flag in particular, would mean the October
run conditioned on a different simulator from the one in the repo now.

In [ ]:
for key in [k for k in T if k.startswith('check_')]:
    chk = T[key]
    print(f'{key[6:]}:')
    diff = chk[chk['ok'] == False]  # noqa: E712  (ok is True / False / empty)
    for net, g in chk.groupby('network'):
        bad = diff[diff['network'] == net]
        print(f'  {net}: {len(bad)} difference(s)' + (': ' + ', '.join(bad['check']) if len(bad) else ''))
    if len(diff):
        display(diff[['network', 'check', 'stored', 'current', 'z']])
for key in [k for k in T if k.startswith('sessions_')]:
    s = T[key]
    print(f'{key}: {int((~s["same"]).sum())} animal(s) whose sessions differ from the current selection')
    if (~s['same']).any():
        display(s[~s['same']])

## 2 · Calibration at an animal's full data (5 sessions)

For 1,000 prior draws per network the truth's rank among 1,000 posterior samples; the curves are the ECDF of the
fractional ranks minus the uniform CDF, with the 95 % simultaneous band (Säilynoja et al. 2022). Inside the band:
calibrated. Above zero in the middle: the posterior is too wide; below: too narrow; a tilt: biased.

In [ ]:
ecdf = T.get('sbc_ecdf')
if ecdf is None:
    print('no sbc_ecdf.csv (sc-sbi-sbc, then sc-validation-tables)')
else:
    nets = ecdf[['rep', 'model']].drop_duplicates().values.tolist()
    params = {m: list(dict.fromkeys(pick(ecdf, model=m)['param'])) for _, m in nets}
    n_col = max(len(v) for v in params.values())
    fig, axes = plt.subplots(len(nets), n_col, figsize=(3.2 * n_col, 2.5 * len(nets)), squeeze=False)
    for i, (rep, model) in enumerate(nets):
        for j, p in enumerate(params[model]):
            plot_sbc_ecdf_diff(pick(ecdf, rep=rep, model=model, n_sessions=5, param=p), ax=axes[i, j])
    plt.tight_layout()

In [ ]:
summ = T.get('sbc_summary')
if summ is not None:
    display(pick(summ, n_sessions=5).pivot_table(index=['rep', 'model'], columns='param', values='ks_p').round(3))
    display(summ.groupby(['rep', 'model', 'n_sessions'])[['n_used', 'n_skipped', 'acceptance_median',
                                                          'budget_hit_frac']].first())

## 3 · Coverage of the posterior intervals

Empirical against nominal coverage of central intervals (50, 80, 90, 95 %); the grey bars are the range a calibrated
network would give with this many draws. These are the intervals reported for real animals.

In [ ]:
cov = T.get('coverage')
if cov is not None:
    nets = cov[['rep', 'model']].drop_duplicates().values.tolist()
    fig, axes = plt.subplots(1, len(nets), figsize=(3.4 * len(nets), 3.2), squeeze=False)
    for j, (rep, model) in enumerate(nets):
        plot_coverage(pick(cov, rep=rep, model=model, n_sessions=5), ax=axes[0, j])
    plt.tight_layout()
    display(pick(cov, nominal=0.9).pivot_table(index=['rep', 'model', 'param'], columns='n_sessions',
                                                values='empirical').round(3))

## 4 · The CV halves (2 and 3 sessions)

The model comparison conditions each network on half an animal's sessions. The networks were trained on 2–6 sessions
with the number of sessions and valid trials as inputs, so the halves are within their training design; the table above
(90 % coverage by session count) and the curves below show whether calibration holds there too.

In [ ]:
if ecdf is not None and (ecdf['n_sessions'] == 2).any():
    nets = ecdf[['rep', 'model']].drop_duplicates().values.tolist()
    fig, axes = plt.subplots(len(nets), n_col, figsize=(3.2 * n_col, 2.5 * len(nets)), squeeze=False)
    for i, (rep, model) in enumerate(nets):
        for j, p in enumerate(params[model]):
            plot_sbc_ecdf_diff(pick(ecdf, rep=rep, model=model, n_sessions=2, param=p), ax=axes[i, j])
    plt.tight_layout()

## 5 · Parameter recovery

From the same SBC draws: RMSE and bias as fractions of the prior width, R² of the posterior median against the truth,
median posterior contraction (0 = posterior as wide as the prior), and the median |z| (about 0.67 when calibrated). A
parameter with contraction near 0 is not identified by these statistics, whatever its R².

In [ ]:
rec = T.get('recovery')
if rec is not None:
    display(pick(rec, n_sessions=5).set_index(['rep', 'model', 'param'])[
        ['rmse_rel', 'bias_rel', 'r2', 'contraction_median', 'abs_z_median', 'frac_abs_z_gt_2']].round(3))
draws = T.get('recovery_draws')
if draws is not None:
    for rep, model in draws[['rep', 'model']].drop_duplicates().values.tolist():
        ps = list(dict.fromkeys(pick(draws, model=model)['param']))
        fig, axes = plt.subplots(2, len(ps), figsize=(3.0 * len(ps), 6.0), squeeze=False)
        for j, p in enumerate(ps):
            d = pick(draws, rep=rep, model=model, n_sessions=5, param=p)
            plot_recovery_scatter(d, ax=axes[0, j])
            plot_contraction(d, ax=axes[1, j])
        plt.tight_layout()

## 6 · Model recovery on the synthetic cohort

Each voter's call (significant at p < 0.05, else no call) and the consensus under both rules, per true model, with
Clopper–Pearson 95 % intervals. `wrong` is the rate that matters most: a significant call for the other model. Grid
search's σ_percep grid stops at 0.30 while the prior runs to 0.50, so its calls are also split by the true σ_percep.

In [ ]:
cs = T.get('call_summary')
if cs is None:
    print('no model-recovery tables (the synthetic run needs grid search, SBI and sc-consensus first)')
else:
    display(cs.set_index(['rule', 'voter', 'true_model'])[
        ['n', 'correct_frac', 'correct_lo', 'correct_hi', 'wrong_frac', 'wrong_lo', 'wrong_hi', 'no_call_frac']].round(3))
    a1, a2 = T['assignments_rule1'], T['assignments_rule2']
    voters = [v for v in ('GS-UM', 'SBI-pooled-UM', 'SBI-moments-UM') if v in a1]
    fig, axes = plt.subplots(1, len(voters) + 2, figsize=(3.4 * (len(voters) + 2), 3.2), squeeze=False)
    for j, v in enumerate(voters):
        calls = a1.assign(call=np.where(a1[f'{v}_p'] < 0.05, a1[v], 'no call'))
        plot_confusion(calls, ax=axes[0, j], call_col='call'); axes[0, j].set_title(v, fontsize=9)
    plot_confusion(a1, ax=axes[0, -2], call_col='Consensus'); axes[0, -2].set_title('consensus, one vote', fontsize=9)
    plot_confusion(a2, ax=axes[0, -1], call_col='Consensus'); axes[0, -1].set_title('consensus, two votes', fontsize=9)
    plt.tight_layout()
    display(T.get('contested'))
    display(T.get('gs_by_sigma'))

In [ ]:
sr = T.get('synthetic_recovery')
if sr is not None:
    print('recovery on the synthetic cohort (real session shapes; saved all-session posteriors):')
    display(sr.set_index(['rep', 'true_model', 'param'])[['n', 'rmse_rel', 'bias_rel', 'r2']].round(3))

## 7 · The real animals

How many real animals' SBI posteriors put σ_percep above 0.30, the grid search's largest value (if several, the grid
search fits them with a truncated grid and its vote is suspect for them), and each animal's call in the October run
and in the current run under both rules.

In [ ]:
rs = T.get('real_sigma')
if rs is None:
    print('no real-run tables (sc-validation-tables --real all-mice/<run id> [--october all-mice/<run id>])')
else:
    above = rs[rs['above_grid']]
    print(f"{above['id'].nunique()} of {rs['id'].nunique()} animals have a posterior median above the grid "
          f"in at least one fit: {', '.join(sorted(above['id'].unique())) or 'none'}")
    display(rs.pivot_table(index='id', columns=['rep', 'model'], values='sigma_percep_median').round(3))
    display(T.get('real_calls'))

## 8 · What to report

Per network: calibration (curves inside the band or not, by parameter) and 90 % coverage at 5 sessions, and whether the
halves keep it. Per parameter: contraction and R²; a parameter that barely contracts is reported as unidentified rather
than estimated. For model identification: per true model, the correct, wrong and no-call rates of the consensus with
their intervals, under the primary rule (one significant vote, contested calls flagged) and the two-vote rule; quote the
wrong-call rate next to the real calls. If the real animals sit above σ_percep 0.30, say so where the grid-search vote
is discussed.